# Día 9 · Unidad 9 — Fuentes de datos (II): CRUD completo con SQLModel

**Objetivos de aprendizaje**
- Crear tablas reales a partir de modelos `SQLModel` (`table=True`) y un `engine`.
- Cargar datos de `data/raw/cartera_polizas.csv` en una base de datos SQLite (Create).
- Consultar registros con `select()`, `.where()`, `and_`/`or_` (Read).
- Modificar registros existentes y confirmar el cambio (Update).
- Eliminar registros de forma segura (Delete) — cerrando el CRUD completo.
- Usar la IA para entender un parámetro de conexión de SQLAlchemy que no conocías, verificándolo contra la documentación oficial.

**Duración estimada:** 90-100 minutos.

**Requisitos previos:** `01_teoria_bbdd_fundamentos.ipynb` de este mismo día.


## 1. De CSV a base de datos: el modelo de datos

Vamos a construir una base de datos con dos tablas conectadas por una foreign key, tal y como vimos en el notebook anterior: **`Asegurado`** (una fila por persona) y **`Poliza`** (una fila por póliza, con una FK al asegurado titular). Los datos de partida son una muestra pequeña de `data/raw/cartera_polizas.csv` — en un caso real cargaríais la cartera completa, pero para seguir el notebook con comodidad basta con unas pocas decenas de filas.


In [ ]:
import pandas as pd

cartera = pd.read_csv("../data/raw/cartera_polizas.csv")
cartera["fecha_alta"] = pd.to_datetime(cartera["fecha_alta"])

# Muestra pequeña y reproducible: las primeras 30 pólizas de la cartera
muestra = cartera.head(30).copy()
print(f"{len(muestra)} pólizas, {muestra['asegurado_id'].nunique()} asegurados distintos")
muestra[["poliza_id", "asegurado_id", "nombre_asegurado", "edad_tomador", "producto", "provincia", "prima_anual", "anulada"]].head()


## 2. Modelos con tabla y foreign key

`table=True` le dice a `SQLModel` que la clase representa una tabla real (no solo un esquema de datos, como en el notebook anterior). `foreign_key="asegurado.id"` conecta `Poliza` con `Asegurado` a través de su primary key — nótese que el nombre de tabla (`asegurado`) es el nombre de la clase en minúsculas por convención de `SQLModel`.


In [ ]:
from typing import Optional
from sqlmodel import SQLModel, Field


class Asegurado(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    asegurado_id: str = Field(index=True, unique=True)  # identificador de negocio (AS20000...)
    nombre: str
    edad: int
    provincia: str


class Poliza(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    poliza_id: str = Field(index=True, unique=True)  # identificador de negocio (POL100000...)
    asegurado_fk: int = Field(foreign_key="asegurado.id")
    producto: str
    prima_anual: float
    anulada: bool


Asegurado, Poliza


## 3. El engine: base de datos en memoria, sin dejar artefactos en el repo

El material original de este bloque creaba un archivo `database.db` en disco con `create_engine("sqlite:///database.db")` — cómodo para inspeccionar el archivo con una herramienta externa, pero deja un artefacto binario que no debería vivir en un repositorio git (y de hecho `.gitignore` ya excluye `*.db` precisamente por este motivo).

Para un notebook de teoría, que se ejecuta de arriba abajo cada vez, es mejor usar SQLite **en memoria**: la URL especial `sqlite://` (sin ruta de archivo) crea una base de datos que vive solo mientras dura el proceso — perfecta para practicar sin dejar residuos.

Hay un detalle importante que conviene conocer bien: **por defecto, cada `Session` que abrís puede obtener una conexión distinta**, y una base de datos en memoria vive únicamente dentro de una conexión — si SQLAlchemy os diera una conexión nueva en cada sesión, cada una vería una base de datos en memoria *distinta y vacía*. Para evitarlo, forzamos que el motor reutilice siempre la **misma** conexión con `poolclass=StaticPool` y `connect_args={"check_same_thread": False}` (necesario porque, en el caso general, cada `Session` podría ejecutarse en un hilo distinto).


In [ ]:
from sqlmodel import create_engine
from sqlalchemy.pool import StaticPool

engine = create_engine(
    "sqlite://",  # SQLite en memoria: no crea ningún archivo .db en disco
    echo=False,   # en el notebook original "echo=True" imprimía cada SQL generado; aquí lo dejamos en False para no saturar el output
    connect_args={"check_same_thread": False},
    poolclass=StaticPool,  # reutiliza siempre la misma conexión -> la BD en memoria persiste entre sesiones
)

SQLModel.metadata.create_all(engine)
print("Tablas creadas:", list(SQLModel.metadata.tables.keys()))


> ⚠️ **Error típico**: usar `sqlite:///:memory:` (o `sqlite://`) con la configuración por defecto del `engine` y sorprenderse de que, tras hacer un `INSERT` en una sesión, una sesión distinta "no ve" los datos. No es un fallo de SQLModel: es que, sin `StaticPool`, cada sesión pudo recibir una conexión (y por tanto una base de datos en memoria) distinta. Con un archivo real (`sqlite:///mi_base.db`) este problema no existe, porque todas las conexiones apuntan al mismo archivo en disco.

### Con ayuda de la IA: entender un parámetro de conexión de SQLAlchemy

`poolclass=StaticPool` y `connect_args={"check_same_thread": False}` no son parámetros evidentes la primera vez que los veis — son justo el tipo de detalle técnico donde pedirle a un asistente de IA una explicación, y **luego verificarla contra la documentación oficial**, es un uso apropiado (`docs/uso_ia_en_el_curso.md`: "lectura de documentación técnica, ejemplos de conexión").


In [ ]:
prompt_conexion = """
Contexto: estoy usando SQLAlchemy/SQLModel en Python con una base de datos
SQLite en memoria (sqlite://) para un notebook de curso. El engine se crea asi:

engine = create_engine(
    "sqlite://",
    connect_args={"check_same_thread": False},
    poolclass=StaticPool,
)

Objetivo: explicame que hace exactamente el parametro poolclass=StaticPool
de SQLAlchemy, que problema resuelve en el caso concreto de SQLite en memoria,
y que hace check_same_thread=False y por que hace falta aqui.

Restricciones: quiero una explicacion tecnica precisa, no una simplificacion
que suene bien pero sea imprecisa. Si hay matices sobre cuando SI hace falta
StaticPool y cuando NO (por ejemplo con un archivo .db real), señalalos.

Formato de salida: explicacion en 3-4 parrafos cortos, mas una frase final
que diga en que casos NO haria falta ninguno de los dos parametros.
"""
print(prompt_conexion)


**Qué se espera obtener**
- Una explicación de que `StaticPool` mantiene una única conexión reutilizada, en vez del comportamiento por defecto de SQLAlchemy (un pool que puede abrir varias conexiones).
- Que en SQLite en memoria cada conexión nueva es, de hecho, una base de datos en memoria *distinta*, y por eso sin `StaticPool` se "pierden" los datos entre sesiones.
- Que `check_same_thread=False` desactiva una comprobación de seguridad del driver `sqlite3` de Python (que por defecto impide usar la misma conexión desde hilos distintos), necesaria porque SQLAlchemy puede gestionar conexiones desde hilos internos.
- Una mención de que, con un archivo `.db` real en disco, `StaticPool` normalmente no hace falta (todas las conexiones ven el mismo archivo).

**Qué debes verificar antes de darlo por bueno**
- Contrasta la afirmación central (una BD SQLite en memoria "desaparece" al cerrar la conexión que la creó) contra la [documentación oficial de SQLAlchemy sobre SQLite](https://docs.sqlalchemy.org/en/20/dialects/sqlite.html#using-a-memory-database-in-multiple-threads) — es un caso muy bien documentado precisamente porque sorprende a mucha gente.
- Compruébalo tú mismo de forma empírica: crea un segundo `engine` en memoria **sin** `poolclass=StaticPool` en una celda nueva, guarda un registro con una sesión y léelo con otra sesión distinta — si el resultado viene vacío, has confirmado el comportamiento con tus propios ojos, no solo porque "la IA lo dijo".
- No aceptes una explicación que generalice de más ("SQLite en memoria siempre necesita StaticPool" es incorrecto si solo abrís una sesión en todo el notebook; el problema aparece específicamente al usar *varias* sesiones/conexiones sobre la misma base en memoria).

*(Comprobación empírica, para no quedarnos solo con la palabra de la IA — fíjate en que el segundo `SELECT`, sin `StaticPool`, no encuentra el registro insertado por el primero):*


In [ ]:
from sqlmodel import Session, select

engine_sin_staticpool = create_engine("sqlite://")  # sin poolclass=StaticPool
SQLModel.metadata.create_all(engine_sin_staticpool)

with Session(engine_sin_staticpool) as s1:
    s1.add(Asegurado(asegurado_id="AS_TEST", nombre="Prueba", edad=30, provincia="test"))
    s1.commit()

with Session(engine_sin_staticpool) as s2:
    encontrados = s2.exec(select(Asegurado)).all()

print(f"Registros visibles desde una sesion distinta, SIN StaticPool: {len(encontrados)}")
print("(si sale 0, confirma justo el comportamiento que explica la IA)")


## 4. Create: cargar la muestra de `cartera_polizas.csv`

Con el `engine` correcto (el que sí usa `StaticPool`), insertamos los asegurados únicos y luego las pólizas, respetando la foreign key: primero tiene que existir el asegurado, para poder referenciarlo desde su póliza.


In [ ]:
with Session(engine) as session:
    # 1) Insertar asegurados únicos de la muestra
    asegurados_unicos = muestra.drop_duplicates(subset="asegurado_id")
    id_por_asegurado_id = {}  # asegurado_id (negocio) -> id (PK autogenerada)

    for _, fila in asegurados_unicos.iterrows():
        asegurado = Asegurado(
            asegurado_id=fila["asegurado_id"],
            nombre=fila["nombre_asegurado"],
            edad=int(fila["edad_tomador"]),
            provincia=fila["provincia"],
        )
        session.add(asegurado)

    session.commit()  # tras el commit, cada "asegurado" ya tiene su id autogenerado

    # session.exec(...) para recuperar el mapeo asegurado_id -> id de PK
    for asegurado in session.exec(select(Asegurado)).all():
        id_por_asegurado_id[asegurado.asegurado_id] = asegurado.id

    # 2) Insertar pólizas, usando la PK del asegurado como foreign key
    for _, fila in muestra.iterrows():
        poliza = Poliza(
            poliza_id=fila["poliza_id"],
            asegurado_fk=id_por_asegurado_id[fila["asegurado_id"]],
            producto=fila["producto"],
            prima_anual=float(fila["prima_anual"]),
            anulada=bool(fila["anulada"]),
        )
        session.add(poliza)

    session.commit()

print(f"{len(id_por_asegurado_id)} asegurados y {len(muestra)} pólizas insertados.")


> ⚠️ **Error típico**: insertar la póliza antes de que exista (o sin haber hecho `commit()` de) su asegurado. La foreign key exige que `asegurado_fk` apunte a un `id` que ya existe en la tabla `asegurado` — si insertáis en el orden equivocado, `SQLModel` lanza un error de integridad referencial en el momento del `commit()`.


## 5. Read: recordatorio rápido (`select`, `.where`, `and_`/`or_`)

Ya vimos esto en el material original — un recordatorio breve antes de pasar a lo nuevo del día.


In [ ]:
from sqlmodel import and_

with Session(engine) as session:
    statement = select(Poliza).where(
        and_(Poliza.producto == "AUTO", Poliza.anulada == False)  # noqa: E712 (comparación explícita con SQLModel)
    )
    polizas_auto_vigentes = session.exec(statement).all()

print(f"Pólizas de AUTO vigentes en la muestra: {len(polizas_auto_vigentes)}")
pd.DataFrame([p.model_dump() for p in polizas_auto_vigentes]).head()


> 📊 En Excel esto sería un `FILTRAR()` (o `AutoFiltro`) sobre la columna `producto` combinado con otro sobre `anulada`; `and_(...)` en SQLModel es el equivalente a apilar dos condiciones de filtro con Y lógico.


## 6. Update: modificar un registro existente

Actualizar con SQLModel sigue siempre el mismo patrón: **leer el objeto**, **modificar sus atributos en memoria** (como cualquier objeto Python normal) y **hacer `commit()`** para que la sesión traduzca el cambio a un `UPDATE` SQL.

Ejemplo de negocio: se detecta que la prima de la póliza `muestra.iloc[0]["poliza_id"]` estaba mal calculada y hay que regularizarla con un incremento del 5%.


In [ ]:
poliza_a_actualizar = muestra.iloc[0]["poliza_id"]

with Session(engine) as session:
    statement = select(Poliza).where(Poliza.poliza_id == poliza_a_actualizar)
    poliza = session.exec(statement).one()  # .one() exige que haya EXACTAMENTE un resultado

    prima_original = poliza.prima_anual
    poliza.prima_anual = round(poliza.prima_anual * 1.05, 2)  # regularización +5%

    session.add(poliza)  # opcional si "poliza" ya viene de esta misma sesión, pero explícito y seguro
    session.commit()
    session.refresh(poliza)  # trae la versión más reciente desde la BD, por si algo la recalculó

    print(f"{poliza_a_actualizar}: {prima_original} € -> {poliza.prima_anual} €")


In [ ]:
# Verificación en una sesión NUEVA, para confirmar que el cambio quedó persistido de verdad
# (no solo en el objeto en memoria de la sesión anterior)
with Session(engine) as session:
    poliza_verificada = session.exec(
        select(Poliza).where(Poliza.poliza_id == poliza_a_actualizar)
    ).one()
    print(f"Prima leída en una sesión nueva: {poliza_verificada.prima_anual} €")


> ⚠️ **Error típico**: modificar un objeto Python (`poliza.prima_anual = ...`) y dar por hecho que ya está guardado, sin llamar a `session.commit()`. El objeto en memoria cambia al instante, pero la base de datos no se entera hasta el `commit()` — por eso el patrón de verificación de la celda anterior (leer en una sesión **nueva**) es la forma correcta de comprobar que un `UPDATE` funcionó de verdad, no solo confiar en el objeto que ya tenéis en la mano.

`.one()` merece una nota aparte: exige que la consulta devuelva **exactamente** un resultado — lanza una excepción tanto si no encuentra ninguno como si encuentra más de uno. Es la forma más segura de leer "el registro que sé que debería existir y ser único" (aquí, por `poliza_id`, que es `unique=True`). Para casos donde puede no existir, se usa `.first()` (devuelve `None` si no hay resultados, en vez de lanzar una excepción).


## 7. Delete: eliminar un registro

Eliminar sigue el mismo patrón: **leer el objeto**, pasarlo a `session.delete(...)`, y `commit()`. Ejemplo de negocio: una póliza se ha dado de baja por error de captura y hay que eliminarla por completo del sistema (distinto de marcar `anulada=True`, que es lo habitual — aquí es literalmente borrar la fila, para practicar el `DELETE`).


In [ ]:
poliza_a_borrar = muestra.iloc[-1]["poliza_id"]  # la última de la muestra

with Session(engine) as session:
    total_antes = len(session.exec(select(Poliza)).all())

    poliza = session.exec(select(Poliza).where(Poliza.poliza_id == poliza_a_borrar)).one()
    session.delete(poliza)
    session.commit()

    total_despues = len(session.exec(select(Poliza)).all())

print(f"Pólizas antes de borrar: {total_antes} | después de borrar: {total_despues}")


In [ ]:
# Confirmar que ya no se puede encontrar (con .first(), que no lanza excepción si no hay resultado)
with Session(engine) as session:
    resultado = session.exec(select(Poliza).where(Poliza.poliza_id == poliza_a_borrar)).first()

print(f"¿Sigue existiendo {poliza_a_borrar}?: {resultado is not None}")


> ⚠️ **Error típico**: llamar a `session.delete(...)` pasando un `poliza_id` (un `str`) en vez del objeto `Poliza` completo. `session.delete()` necesita la **instancia** que la sesión está siguiendo, no el identificador — por eso siempre se lee primero con `select()`/`.where()` y se borra el objeto resultante, nunca "por id a pelo" como haríais en SQL crudo (`DELETE FROM poliza WHERE poliza_id = ...`).

En un sistema real de seguros, borrar una póliza casi nunca es la operación correcta — normalmente se prefiere un *soft delete* (marcar `anulada=True` y conservar el histórico, tal y como hace el propio dataset `cartera_polizas.csv` con su columna `anulada`) para no perder trazabilidad. Pero conocer el `DELETE` real es necesario para casos legítimos: datos duplicados por error, pruebas, corrección de una captura equivocada.

**Para ti:** añade un nuevo asegurado con `Asegurado(asegurado_id="AS_NUEVO", nombre="Jordi Camps", edad=29, provincia="girona")`, dale de alta una póliza de producto `"HOGAR"` con prima `310.0`, y después actualiza esa misma póliza para marcarla `anulada=True`. Verifica el resultado final con una consulta `select(Poliza).where(Poliza.asegurado_fk == ...)` en una sesión nueva.


In [ ]:
# Tu código aquí: CREATE (asegurado + póliza), luego UPDATE (anulada=True), luego verificación


## Resumen

- `table=True` convierte un modelo `SQLModel` en una tabla real; `foreign_key="tabla.columna"` la conecta con otra tabla por su primary key.
- Una base de datos SQLite en memoria (`sqlite://`) es ideal para practicar sin dejar artefactos `.db` en el repositorio — pero necesita `poolclass=StaticPool` (+ `check_same_thread=False`) si vais a abrir más de una `Session`, porque de lo contrario cada conexión ve una base en memoria distinta y vacía.
- **Create**: instanciar el modelo, `session.add(...)`, `session.commit()`. Respetad el orden cuando hay foreign keys: primero la tabla "padre" (aquí, `Asegurado`), luego la "hija" (`Poliza`).
- **Read**: `select(Modelo).where(condición)`, con `and_`/`or_` para combinar condiciones — ya lo vimos en detalle en el material original de la unidad.
- **Update**: leer el objeto, modificar sus atributos como un objeto Python normal, `session.commit()`. El cambio no existe en la base de datos hasta el `commit()`, aunque el objeto en memoria ya lo refleje.
- **Delete**: leer el objeto con `select()`/`.where()`, `session.delete(objeto)`, `session.commit()`. No se puede borrar "por id a pelo": hace falta la instancia.
- `.one()` exige exactamente un resultado (lanza excepción si hay 0 o más de 1); `.first()` devuelve `None` si no hay resultados, sin lanzar excepción — elegid según si la ausencia del registro es un caso normal o un error.
- Un asistente de IA es útil para entender parámetros de conexión poco habituales (`poolclass`, `check_same_thread`...), pero la explicación se acepta solo tras verificarla contra la documentación oficial o, mejor aún, comprobándola empíricamente vosotros mismos.

**Siguiente:** `03_teoria_apis_rest.ipynb`, la otra mitad de la Unidad 9 — consumir una API REST con `requests`, credenciales y variables de entorno.
